# Introduction to sentiment analysis with NLI

In this notebook, you will implement a sentiment scoring pipeline using a Natural Language Inference (NLI) model.
You will analyze whether news headlines about S&P 500 companies have positive or negative sentiment and explore how this sentiment correlates with market returns.


## Install and Import librairies
Install necessary packages and import the required libraries for:
- Loading data
- Using transformer models
- Plotting and visualizing results
- Downloading financial data

In [3]:
%pip install hf_xet
%pip install yfinance
%pip install transformers
%pip install torch
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 2.4/241.3 MB 16.4 MB/s eta 0:00:15
   - -------------------------------------- 6.8/241.3 MB 19.0 MB/s eta 0:00:13
   - -------------------------------------- 12.1/241.3 MB 21.2 MB/s eta 0:00:11
   -- ------------------------------------- 17.6/241.3 MB 22.7 MB/s eta 0:00:10
   --- ------------------------------------ 20.7/241.3 MB 21.1 MB/s eta 0:00:11
   ---- ----------------------------------- 25.7/241.3 MB 21.4 MB/s eta 0:00:11
   ----- ---------------------------------- 31.2/241.3 MB 22.4 MB/s eta 0:00:10
   ----- ---------------------------------- 36.2/241


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------- ----------------------- 3.4/8.1 MB 19.7 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.1 MB 20.5 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 19.2 MB/s eta 0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 17.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   --------------------------- ------------ 4.7/7.0 MB 24.0 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 23.7 MB/s eta 0:00:00
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import math
import matplotlib.pyplot as plt
import yfinance as yf

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

## Load news data
Load two datasets:
- `df_news.csv`: contains headlines and summaries
- `df_metadata.csv`: contains ticker symbols and company sector info

We drop duplicate summaries to avoid redundant sentiment scoring.

In [7]:
df_news = pd.read_csv('C:/Users/jfmag/OneDrive/Documentos/Maestría/Fintech/df_news.csv')
df_news.drop_duplicates('SUMMARY', inplace=True)
display(df_news)

,TICKER,TITLE,SUMMARY,PUBLICATION_DATE,PROVIDER,URL
0,MMM,2 Dow Jones Stocks with Promising Prospects an...,The Dow Jones (^DJI) is made up of 30 of the m...,2025-05-29 04:33:58+00:00,StockStory,https://finance.yahoo.com/news/2-dow-jones-sto...
1,MMM,3 S&P 500 Stocks Skating on Thin Ice,The S&P 500 (^GSPC) is often seen as a benchma...,2025-05-27 04:34:42+00:00,StockStory,https://finance.yahoo.com/news/3-p-500-stocks-...
2,MMM,3M Rises 15.8% YTD: Should You Buy the Stock N...,"MMM is making strides in the aerospace, indust...",2025-05-22 14:08:00+00:00,Zacks,https://finance.yahoo.com/news/3m-rises-15-8-y...
3,MMM,Q1 Earnings Roundup: 3M (NYSE:MMM) And The Res...,Quarterly earnings results are a good time to ...,2025-05-22 03:31:21+00:00,StockStory,https://finance.yahoo.com/news/q1-earnings-rou...
4,MMM,3 Cash-Producing Stocks with Questionable Fund...,While strong cash flow is a key indicator of s...,2025-05-19 04:41:32+00:00,StockStory,https://finance.yahoo.com/news/3-cash-producin...
...,...,...,...,...,...,...
4866,ZTS,2 Dividend Stocks to Buy With $500 and Hold Fo...,Zoetis is a leading animal health company with...,2025-05-23 10:30:00+00:00,Motley Fool,https://www.fool.com/investing/2025/05/23/2-di...
4867,ZTS,Zoetis (NYSE:ZTS) Declares US$0.50 Dividend Pe...,Zoetis (NYSE:ZTS) recently affirmed a dividend...,2025-05-22 17:49:43+00:00,Simply Wall St.,https://finance.yahoo.com/news/zoetis-nyse-zts...
4868,ZTS,Jim Cramer on Zoetis (ZTS): “It Does Seem to B...,We recently published a list of Jim Cramer Tal...,2025-05-21 18:14:38+00:00,Insider Monkey,https://finance.yahoo.com/news/jim-cramer-zoet...
4869,ZTS,Zoetis (ZTS) Upgraded to Buy: Here's Why,Zoetis (ZTS) might move higher on growing opti...,2025-05-21 16:00:08+00:00,Zacks,https://finance.yahoo.com/news/zoetis-zts-upgr...


In [8]:
df_meta = pd.read_csv('C:/Users/jfmag/OneDrive/Documentos/Maestría/Fintech/df_metadata-1.csv')
display(df_meta)

,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,3M Company,Industrials,Conglomerates
1,AOS,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,Healthcare,Medical Devices
3,ABBV,AbbVie Inc.,Healthcare,Drug Manufacturers - General
4,ACN,Accenture plc,Technology,Information Technology Services
...,...,...,...,...
485,XEL,Xcel Energy Inc.,Utilities,Utilities - Regulated Electric
486,XYL,Xylem Inc.,Industrials,Specialty Industrial Machinery
487,YUM,"Yum! Brands, Inc.",Consumer Cyclical,Restaurants
488,ZBH,"Zimmer Biomet Holdings, Inc.",Healthcare,Medical Devices


## Sentiment Analysis with NLI

In this section, you will apply a CrossEncoder NLI model (`cross-encoder/nli-deberta-v3-base`) to estimate sentiment from news headlines.

👉 **Instructions**:

1. Use a CrossEncoder NLI model to compute how much a news headline implies a **positive** or **negative** sentiment.
2. For each news title, compute the probability of it being **positive** and **negative**, and store them in `POSITIVE_PROB` and `NEGATIVE_PROB`.
3. Derive a final sentiment score by subtracting: `SENTIMENT = POSITIVE_PROB - NEGATIVE_PROB`.

✅ This score will serve as your sentiment signal, ranging from negative to positive.

> ℹ️ You are free to decide how to structure the input pairs and how to apply the model.


In [10]:
# CODE HERE
# Use as many coding cells as you need

model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/nli-deberta-v3-base')
tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-base')

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
headlines_list=df_news['TITLE'].tolist()
positive_headline='This is a positive headline'
negative_headline='This is a Negative headline'

In [ ]:
headlines_pos= [(headline, positive_headline)for headline in headlines_list]
headlines_neg= [(headline, negative_headline)for headline in headlines_list]

In [ ]:
features_pos = tokenizer(*zip(*headlines_pos), padding=True, truncation=True, return_tensors='pt')
features_neg = tokenizer(*zip(*headlines_neg), padding=True, truncation=True, return_tensors='pt')

In [ ]:
model.eval()
with torch.no_grad():
    logits_pos = model(**features_pos).logits
    probs_pos = torch.softmax(logits_pos, dim=1)
    positive_prob = probs_pos[:, 1]  # POSITIVE_PROB

with torch.no_grad():
    logits_neg = model(**features_neg).logits
    probs_neg = torch.softmax(logits_neg, dim=1)
    negative_prob = probs_neg[:, 1]  # NEGATIVE_PROB

sentiment = positive_prob - negative_prob


In [ ]:
print(sentiment)

## Compare Sentiment with Returns

In this section, you'll explore how daily news sentiment aligns with market behavior.

👉 **Instructions**:

1. Group the news data by **publication date** and compute the **average sentiment per day**.
2. Download **daily stock prices** for the relevant tickers using `yfinance`.
3. Compute **daily returns** and use their average as a proxy for the market (e.g., S\&P 500).
4. Visualize both **daily sentiment** and **daily returns** over time using line plots.
5. Create a **dual y-axis chart** to compare trends more effectively.

✅ This section helps you assess whether changes in sentiment coincide with market movements.

> ℹ️ Focus on trend relationships, not just visual similarity—this is an opportunity to start thinking about predictive signals.

In [ ]:
# CODE HERE
# Use as many coding cells as you need
df_sentiments= df_news[['PUBLICATION_DATE','TITLE']]
df_sentiments['PUBLICATION_DATE'] = pd.to_datetime(df_sentiments['PUBLICATION_DATE']).dt.date
df_sentiments['SENTIMENT']=sentiment
df_sentiments.set_index('PUBLICATION_DATE', inplace=True)
df_sentiments

In [ ]:
daily_avg_sent=df_sentiments.groupby('PUBLICATION_DATE')['SENTIMENT'].mean()
daily_avg_sent

In [ ]:
#relevant tickers by random sample
rand_tickers=df_meta['TICKER'].sample(50).tolist()
rand_tickers

In [ ]:
s_dt=pd.to_datetime(df_news['PUBLICATION_DATE'].min()).date()
print(s_dt)
e_dt=pd.to_datetime(df_news['PUBLICATION_DATE'].max()).date()
print(e_dt)

In [ ]:
df_stocks= yf.download(rand_tickers, start=s_dt, end=e_dt)['Close']
df_stocks

In [ ]:
df_returns=df_stocks.pct_change()
df_returns

In [ ]:
df_returns['DAILY_AVG_RETURN']=df_returns.mean(axis=1)
df_returns

In [ ]:
df_chart=pd.concat([daily_avg_sent, df_returns['DAILY_AVG_RETURN']], axis=1)
plt.figure(figsize=(10, 6))
plt.plot(df_chart.index, df_chart['SENTIMENT'], label='Sentiment')
plt.plot(df_chart.index, df_chart['DAILY_AVG_RETURN'], label='Daily Average Return')
plt.title('Sentiment vs. Daily Average Return')
plt.legend()
plt.show()

In [ ]:
#dual y axis plot

x = df_chart.index
y1 = df_chart['SENTIMENT']
y2 = df_chart['DAILY_AVG_RETURN']
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(x, y1, color='blue', label='Primary Data')
ax1.set_xlabel('date')
ax1.set_ylabel('SENTIMENT', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(x, y2, color='red', label='Secondary Data')
ax2.set_ylabel('DAILY_AVG_RETURN', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('SENTIMENT vs. DAILY_AVG_RETURN')
plt.show()

## Compare Daily Portfolio Value Against Daily Average Sentiment

In this section, you will simulate a simple market portfolio and explore how its performance aligns with daily sentiment scores.

👉 **Instructions**:

1. Simulate a **market portfolio** by computing the cumulative return of the average daily return across all tickers.
2. Start the portfolio with an **initial value of 1.0** and track its value over time.
3. Plot the **daily average sentiment** and the **portfolio value** using a dual-axis line chart.

✅ This visualization lets you explore whether market sentiment leads or lags behind portfolio movements.

> ℹ️ Think about how this setup could inform a basic trading strategy—or whether sentiment could serve as a timing signal.


In [ ]:
# CODE HERE
# Use as many coding cells as you need
#-----------Step 1---------------
df_retruns_all=df_stocks.pct_change()
df_returns_all=df_retruns_all.mean(axis=1)
df_cum_value = (1 + df_returns_all).cumprod()
#------------Step 2---------------
df_cum_value.iloc[0] = 1.0
df_cum_value

In [ ]:
df_sent=pd.concat([daily_avg_sent, df_cum_value], axis=1)
df_sent

In [ ]:
x = df_sent.index
y1 = df_sent['SENTIMENT']
y2 = df_sent[0]
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(x, y1, color='blue', label='Primary Data')
ax1.set_ylabel('SENTIMENT', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(x, y2, color='red', label='Secondary Data')
ax2.set_ylabel('CUMULATIVE RETRUNS', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('SENTIMENT vs. CUMULATIVE RETRUNS')
plt.show()

According to the chart, the trading strategy should be swing trading, which means to hold positons for days or weeks. From 2024-07 the strategy should be only buy postions, then on Jan 2025 we should change to selling postions, or we can just wait to start sellinng on March 2025. Lastly on Apr 2025 we should take again buying postions only.

## Compute Sector-Level Sentiment and Returns

In this section, you’ll analyze how sector-level news sentiment compares to market performance for May. You’ll also measure whether sentiment correlates with returns.

👉 **Instructions**:

1. Group companies by **sector** using the metadata.
2. For each sector:

   * Compute **monthly average sentiment** (Feb to May).
   * (Optional) Count the number of headlines per month.
3. Compute **monthly stock returns** from price data and extract **May returns**.
4. Build a comparison table with:

   * `SECTOR`
   * `SENTIMENT` (May sentiment)
   * `RETURN` (May return)
5. Compute and print the **correlation** between May sentiment and May returns.

✅ This step helps evaluate whether optimistic news coverage for a sector is associated with better performance.

> 💡 Once your analysis is complete, consider:
>
> * Which sectors *looked* good in the news but didn’t perform?
> * Which sectors performed well despite neutral/negative sentiment?
> * Would you invest based on sentiment alone? Why or why not?



In [ ]:
# CODE HERE
# Use as many coding cells as you need
df_news_sector=pd.merge(df_news, df_meta, on='TICKER')
df_news_sector['PUBLICATION_DATE']=pd.to_datetime(df_news_sector['PUBLICATION_DATE']).dt.date
df_news_sector

In [ ]:
s_dt=pd.to_datetime(df_news_sector['PUBLICATION_DATE'].min()).date()
print(s_dt)
e_dt=pd.to_datetime(df_news_sector['PUBLICATION_DATE'].max()).date()
print(e_dt)

In [ ]:
df_feb_may=df_news_sector[(df_news_sector['PUBLICATION_DATE'] >= pd.to_datetime('2025-02-01').date()) & (df_news_sector['PUBLICATION_DATE'] <= pd.to_datetime('2025-05-31').date())]
df_feb_may

In [ ]:
headlines_list=df_feb_may['TITLE'].tolist()
positive_headline='This is a positive headline'
negative_headline='This is a Negative headline'
headlines_pos= [(headline, positive_headline)for headline in headlines_list]
headlines_neg= [(headline, negative_headline)for headline in headlines_list]
features_pos = tokenizer(*zip(*headlines_pos), padding=True, truncation=True, return_tensors='pt')
features_neg = tokenizer(*zip(*headlines_neg), padding=True, truncation=True, return_tensors='pt')
model.eval()
with torch.no_grad():
    logits_pos = model(**features_pos).logits
    probs_pos = torch.softmax(logits_pos, dim=1)
    positive_prob = probs_pos[:, 1]  # POSITIVE_PROB

with torch.no_grad():
    logits_neg = model(**features_neg).logits
    probs_neg = torch.softmax(logits_neg, dim=1)
    negative_prob = probs_neg[:, 1]  # NEGATIVE_PROB

sentiment = positive_prob - negative_prob

In [ ]:
df_feb_may['SENTIMENT']=sentiment
df_feb_may

In [ ]:
df_feb_may['MONTH'] = pd.to_datetime(df_feb_may['PUBLICATION_DATE']).dt.month
monthly_avg_sentiment = df_feb_may.groupby(['MONTH'])['SENTIMENT'].mean()
monthly_avg_sentiment

In [ ]:
headline_counts = (
    df_feb_may.groupby(['MONTH'])['TITLE']
    .count()
    .reset_index(name="Headlines Count")
)
headline_counts.set_index('MONTH', inplace=True)
headline_counts

In [ ]:
#retrieve stock data
df_stocks= yf.download(df_feb_may['TICKERS'].unique(), start='2025-02-01', end='2025-05-31')['Close']
df_stocks

In [ ]:
#Computing stock returns
df_returns=df_stocks.pct_change()
#computing monthly returns
df_returns['MONTH'] = pd.to_datetime(df_returns.index).month
monthly_returns = df_returns.groupby('MONTH').mean()
monthly_returns
#retrieve may data only
may_returns=monthly_returns.iloc[4]
may_returns

In [ ]:
#comparative table for may only
df_comptbl= df_meta[['SECTOR', 'TICKER']]
df_comptbl['SENTIMENT']=monthly_avg_sentiment.loc[5]
df_comptbl['RETURN']=may_returns
df_comptbl

In [ ]:
#may correlation
corr=df_comptbl['SENTIMENT'].corr(df_comptbl['RETURN'])
corr

### **Question 1.** Which sectors *looked* good in the news? How did they perform?


YOUR WRITTEN RESPONSE HERE




### **Question 2.** Which sectors performed well despite neutral/negative sentiment?


YOUR WRITTEN RESPONSE HERE

### **Question 3.**  Would you invest based on sentiment alone? Why or why not?

YOUR WRITTEN RESPONSE HERE


### **Question 4.**  How would you go about testing a sentiment analysis strategy in a more robust way?

YOUR WRITTEN RESPONSE HERE


In [ ]:
#%pip install --upgrade --force-reinstall pandas transformers torch matplotlib yfinance